# Lab 13: Toy Diffusion Models

            **Duration:** 3 hours  
            **Lecture alignment:** Week 13 — Denoising diffusion fundamentals  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Implement a forward Gaussian-noising schedule.
- Train a timestep-conditioned network to predict noise.
- Run and visualize an approximate reverse-diffusion sampler.

            ## Three-hour activity plan

            - 0–30 min: variance schedule and forward process
- 30–65 min: timestep conditioning and loss
- 65–120 min: train noise predictor
- 120–160 min: reverse sampler and trajectories
- 160–180 min: compare assumptions, checks, and reflection


## Book grounding

            - Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20273
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_13")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_13"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 13, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

At which forward timestep will the four clusters become visually indistinguishable? Predict whether low noise-prediction MSE guarantees good samples.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Forward diffusion on a multimodal distribution


In [ ]:
centers=torch.tensor([[-1.5,-1.0],[-1.5,1.0],[1.5,-1.0],[1.5,1.0]])
def sample_data(n):
    ids=torch.randint(0,4,(n,));return centers[ids]+.18*torch.randn(n,2)
T=30;betas=torch.linspace(1e-4,.08,T,device=DEVICE);alphas=1-betas;alpha_bar=torch.cumprod(alphas,0)
def q_sample(x0,t,noise=None):
    if noise is None:noise=torch.randn_like(x0)
    ab=alpha_bar[t].unsqueeze(1);return ab.sqrt()*x0+(1-ab).sqrt()*noise,noise
x0=sample_data(800).to(DEVICE);snapshots=[]
for step in [0,7,15,29]:snapshots.append(q_sample(x0,torch.full((len(x0),),step,device=DEVICE,dtype=torch.long))[0].cpu())


## Activity 2 — Train a timestep-conditioned noise predictor


In [ ]:
class NoisePredictor(nn.Module):
    def __init__(self):super().__init__();self.net=nn.Sequential(nn.Linear(3,64),nn.SiLU(),nn.Linear(64,64),nn.SiLU(),nn.Linear(64,2))
    def forward(self,x,t):return self.net(torch.cat([x,t.float().unsqueeze(1)/(T-1)],1))
model=NoisePredictor().to(DEVICE);opt=torch.optim.Adam(model.parameters(),lr=.003);losses=[]
for _ in range(350 if FAST_MODE else 2200):
    clean=sample_data(128).to(DEVICE);t=torch.randint(0,T,(len(clean),),device=DEVICE);noisy,noise=q_sample(clean,t)
    opt.zero_grad();loss=F.mse_loss(model(noisy,t),noise);loss.backward();opt.step();losses.append(loss.item())
print({"early_mean_loss":float(np.mean(losses[:50])),"late_mean_loss":float(np.mean(losses[-50:]))})


## Activity 3 — Reverse sampling and denoising trajectory


In [ ]:
@torch.no_grad()
def reverse_sample(n=800):
    x=torch.randn(n,2,device=DEVICE);trajectory={T:x.cpu()}
    for step in reversed(range(T)):
        t=torch.full((n,),step,device=DEVICE,dtype=torch.long);pred_noise=model(x,t)
        mean=(x-betas[step]/torch.sqrt(1-alpha_bar[step])*pred_noise)/torch.sqrt(alphas[step])
        x=mean+(torch.sqrt(betas[step])*torch.randn_like(x) if step>0 else 0)
        if step in (20,10,0):trajectory[step]=x.cpu()
    return x.cpu(),trajectory
generated,trajectory=reverse_sample()
fig,axes=plt.subplots(2,4,figsize=(12,6))
for ax,data,step in zip(axes[0],snapshots,[0,7,15,29]):ax.scatter(data[:,0],data[:,1],s=4,alpha=.35);ax.set_title(f"forward t={step}")
for ax,(step,data) in zip(axes[1],sorted(trajectory.items(),reverse=True)):ax.scatter(data[:,0],data[:,1],s=4,alpha=.35);ax.set_title(f"reverse t={step}")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"diffusion_trajectory.png",dpi=150);plt.show()
torch.save(model.state_dict(),ARTIFACT_DIR/"noise_predictor.pt")


## Automated checks


In [ ]:
assert len(snapshots)==4 and all(x.shape==(800,2) for x in snapshots)
assert generated.shape==(800,2) and torch.isfinite(generated).all()
assert np.mean(losses[-50:])<np.mean(losses[:50])
assert (ARTIFACT_DIR/"diffusion_trajectory.png").exists()
print("All Lab 13 checks passed.")


## Deliverables

                - Forward-process snapshots
- Noise-prediction learning evidence
- Reverse-sampling trajectory and saved predictor
- Brief comparison with VAE and GAN sampling

                Submit the executed notebook and the files created in `/content/artifacts/lab_13/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: reduce reverse steps by selecting a sparse timestep schedule and compare speed/quality.")
else:
    print("Extension disabled: conditional denoising or a reduced-step DDIM-style sampler.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
